# Mean field of the faculae (sweep_facula_B.dag)

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sunsim import observations as obs

BASE_FB = '/data/spotless/scratch/sstucki/starsim/facula_B'
TAG = 'facula_B_m45s_size_4096_N60'
SETS = os.path.join(BASE_FB, 'sets_' + TAG)
START = '2015-07-18'                             # t = 0 of the simulations
OBS = '/data/spotless/scratch/sstucki/starsim/observations'
HARPSN = os.path.join(OBS, 'harpn_sun_release_timeseries_2015-2025.csv')
INTERVENTIONS = os.path.join(OBS, 'harpsn_intervention_days.txt')
SORCE, TSIS = os.path.join(OBS, 'DataSSI - SORCE.sav'), os.path.join(OBS, 'tsis_tsi_24hr.csv')
MAX_FF_FC, MAX_FF_SP = 20.0, 5.0                 # epochs left out (bad HMI images), as obs.sweep_stats

METRICS = [('rv_rms', 'RV - HARPS-N', 'm/s'), ('fwhm_rms', 'FWHM - HARPS-N', 'm/s'), ('bis_rms', 'BIS - HARPS-N', 'm/s'),
           ('sorce_rms', 'TSI - SORCE/TIM', 'ppm'), ('tsis_rms', 'TSI - TSIS-1/TIM', 'ppm')]

df = obs.load_sweep_stats(SETS).sort_values('B_fc').reset_index(drop=True)
METRICS = [m for m in METRICS if m[0] in df and df[m[0]].notna().any()]
print(len(df), 'sets, B_fc =', ', '.join('%g' % b for b in df['B_fc']))
df[['B_fc'] + [m[0] for m in METRICS] + [m[0].replace('_rms', '_mad') for m in METRICS]].round(3)

In [ ]:
plt.rcParams.update({
    'font.size': 14,
    'axes.labelsize': 16,
    'axes.titlesize': 18,
    'xtick.labelsize': 14,
    'ytick.labelsize': 14,
    'legend.fontsize': 12,
    'figure.titlesize': 18,
})
try:
    import cmasher as cmr
    CMAP = cmr.get_sub_cmap(cmr.torch, 0, 0.95)
except ImportError:
    CMAP = 'inferno'

## Heatmap of the sweep

In [ ]:
def sweep_heatmap(df, metrics, kind='rms', save=None):
    """RMS (or MAD) of every metric (rows) against B_fc (columns), coloured by the value over the minimum of its row;
    the values in the cells, the best B_fc of every row outlined in red."""
    B = df['B_fc'].values
    vals = np.array([df[k.replace('_rms', '_' + kind)].values for k, _, _ in metrics], dtype=float)
    rel = vals / np.nanmin(vals, axis=1, keepdims=True)
    fig, ax = plt.subplots(figsize=(1.6 * len(B) + 3, 0.9 * len(metrics) + 1.5))
    im = ax.imshow(rel, aspect='auto', cmap=CMAP, vmin=1, vmax=np.nanmax(rel))
    for i in range(len(metrics)):
        for j in range(len(B)):
            if np.isfinite(vals[i, j]):
                ax.text(j, i, '%.3g' % vals[i, j], ha='center', va='center', fontsize=12,
                        color='w' if im.norm(rel[i, j]) < 0.6 else 'k')
        j = np.nanargmin(vals[i])
        ax.add_patch(plt.Rectangle((j - 0.5, i - 0.5), 1, 1, fill=False, ec='red', lw=3))
    ax.set_xticks(range(len(B)))
    ax.set_xticklabels(['%g' % b for b in B])
    ax.set_yticks(range(len(metrics)))
    ax.set_yticklabels(['%s [%s]' % (t, u) for _, t, u in metrics])
    ax.set_xlabel(r'$B_{fc}$ [G]')
    cbar = fig.colorbar(im, ax=ax, fraction=0.03, pad=0.02)
    cbar.set_label('%s / minimum of the row' % kind.upper())
    if save:
        fig.savefig(save, bbox_inches='tight')
    return fig, ax


sweep_heatmap(df, METRICS, 'rms')
plt.show()
sweep_heatmap(df, METRICS, 'mad')
plt.show()

## RMS against the mean field of the faculae

In [ ]:
fig, axes = plt.subplots(1, len(METRICS), figsize=(4 * len(METRICS), 3.8), sharex=True, squeeze=False)
for ax, (k, title, unit) in zip(axes[0], METRICS):
    ax.plot(df['B_fc'], df[k], 'o-', c='k', label='RMS')
    ax.plot(df['B_fc'], df[k.replace('_rms', '_mad')], 's--', c='0.5', label='MAD')
    ax.axvline(200, c='0.7', lw=1, ls=':')
    ax.set_title(title)
    ax.set_xlabel(r'$B_{fc}$ [G]')
    ax.set_ylabel('[%s]' % unit)
axes[0, 0].legend()
fig.tight_layout()
plt.show()

## Time series

In [ ]:
def series(folder):
    """Time series of one set (obs._read_series), the bad epochs left out."""
    s = obs._read_series(os.path.join(SETS, folder))
    keep = ~((s['ff_fc'] > MAX_FF_FC) | (s['ff_sp'] > MAX_FF_SP))
    return {k: v[keep] for k, v in s.items()}


colours = plt.get_cmap('viridis')(np.linspace(0, 0.9, len(df)))
fig, axes = plt.subplots(2, 1, figsize=(15, 8), sharex=True)
for c, (_, r) in zip(colours, df.iterrows()):
    s = series(r['folder'])
    axes[0].plot(s['time'], 1e6 * (s['lc'] / np.nanmean(s['lc']) - 1), '.', ms=1.5, c=c,
                 label=r'$B_{fc}$ = %g G' % r['B_fc'])
    axes[1].plot(s['time'], s['rv'] - np.nanmedian(s['rv']), '.', ms=1.5, c=c)
axes[0].set_ylabel('Flux [ppm]')
axes[1].set_ylabel('RV [m/s]')
axes[1].set_xlabel('Time [days since %s]' % START)
axes[0].legend(markerscale=6, ncol=len(df), fontsize=10)
fig.tight_layout()
plt.show()

## Best sets against the observations

In [ ]:
harps = obs.load_harpsn(HARPSN, START)
sorce, tsis = obs.load_sorce(SORCE, START), obs.load_tsis(TSIS, START)
cuts = obs.load_interventions(INTERVENTIONS, START)
for k in ('rv_rms', 'tsis_rms'):
    b = obs.best_sets(df, k, 1).iloc[0]
    print('best %s: B_fc = %g G (%.4g)' % (k, b['B_fc'], b[k]))
    s = series(b['folder'])
    ph = [obs.compare_photometry(sorce, s['time'], s['lc']), obs.compare_photometry(tsis, s['time'], s['lc'])]
    rv = obs.compare_rv(harps, s['time'], s['rv'], interventions=cuts)
    obs.print_summary(ph, rv)
    obs.plot_comparison(ph, rv, start=START)
    plt.show()